Imports, paths, helpers

In [1]:
# === Setup ===
import os, re, json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# lightweight sentiment library
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

FINAL = Path("data/final")
SENT_DIR = Path("data/sentiment")
FINAL.mkdir(parents=True, exist_ok=True)
SENT_DIR.mkdir(parents=True, exist_ok=True)

def to_text(x): 
    try: return str(x)
    except: return ""

def clean_text(s: str) -> str:
    s = to_text(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


Load reviews safely

In [2]:
# === Load reviews (dynamic output from Arjun's pipeline) ===
rev_path = FINAL / "reviews.csv"
assert rev_path.exists(), f"Missing {rev_path}"

reviews = pd.read_csv(rev_path)
# Ensure expected columns exist
for c in ["review_id","place_id","text","rating","lang","publish_time_utc","name","lat","lng","category"]:
    if c not in reviews.columns:
        reviews[c] = None

# Clean & filter
reviews["text"] = reviews["text"].apply(clean_text)
reviews = reviews[reviews["text"] != ""].copy()

# Optional: keep English-ish
if "lang" in reviews.columns:
    reviews = reviews[reviews["lang"].fillna("").astype(str).str.startswith(("en","EN","En")) | (reviews["lang"].isna())]

print("Loaded reviews:", reviews.shape)
reviews.head(2)


Loaded reviews: (27103, 13)


,review_id,place_id,text,rating,lang,publish_time_utc,author_name,review_photo_url,review_time,name,lat,lng,category
0,1cc99ebc14f41739906d0f99,ChIJ1QL4cyBdDW0RcU21rEmrs2A,Had a 2 day intensive team building camp. Grea...,5,en,1708136374,Andy Parr,NaN,1708136374,None,None,None,None
1,03e683633d3b2d748e132486,ChIJ1QL4cyBdDW0RcU21rEmrs2A,We are still here just absolutely lovely,5,en,1704774509,Katherine Taniwha,NaN,1704774509,None,None,None,None


Compute sentiment (VADER with rating fallback)

In [3]:
# === Sentiment scoring ===
def score_sentiment(row):
    txt = row.get("text","") or ""
    vs = analyzer.polarity_scores(txt)  # {'neg':..,'neu':..,'pos':..,'compound':..}
    comp = vs["compound"]  # -1..+1

    # Optional: blend a little rating signal if present
    r = row.get("rating", None)
    if pd.notna(r):
        try:
            r = float(r)  # 1..5
            # map rating into -0.5..+0.5 and blend lightly
            comp = 0.8*comp + 0.2*((r-3)/2)
        except:
            pass

    if comp >= 0.25:
        label = "positive"
    elif comp <= -0.25:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series({
        "compound": round(float(comp),4),
        "sentiment_label": label
    })

sent = reviews.apply(score_sentiment, axis=1)
reviews_scored = pd.concat([reviews, sent], axis=1)

print("Scored:", reviews_scored.shape)
reviews_scored[["text","compound","sentiment_label"]].head(3)


Scored: (27103, 15)


,text,compound,sentiment_label
0,Had a 2 day intensive team building camp. Grea...,0.8325,positive
1,We are still here just absolutely lovely,0.6992,positive
2,We’ll it ready depends what you are looking fo...,0.7815,positive


Per-place aggregation (totals and percentages)

In [4]:
# === Aggregate per place (for map/summary) ===
def pct(pos, total): 
    return round(100.0 * pos / total, 2) if total else 0.0

grp = (reviews_scored
       .groupby(["place_id","name","lat","lng","category"], dropna=False, as_index=False)
       .agg(
            review_count=("review_id","nunique"),
            avg_rating=("rating","mean"),
            avg_compound=("compound","mean"),
            pos_count=("sentiment_label", lambda s: (s=="positive").sum()),
            neg_count=("sentiment_label", lambda s: (s=="negative").sum()),
            neu_count=("sentiment_label", lambda s: (s=="neutral").sum()),
       ))

grp["pct_positive"] = grp.apply(lambda r: pct(r["pos_count"], r["review_count"]), axis=1)
grp["pct_negative"] = grp.apply(lambda r: pct(r["neg_count"], r["review_count"]), axis=1)
grp["pct_neutral"]  = grp.apply(lambda r: pct(r["neu_count"], r["review_count"]), axis=1)

# overall label per place
def overall_label(row):
    if row["pct_positive"] >= 60: return "positive"
    if row["pct_negative"] >= 40: return "negative"
    return "neutral"

grp["overall_label"] = grp.apply(overall_label, axis=1)

print("Place-level summary:", grp.shape)
grp.head(3)


Place-level summary: (5373, 15)


,place_id,name,lat,lng,category,review_count,avg_rating,avg_compound,pos_count,neg_count,neu_count,pct_positive,pct_negative,pct_neutral,overall_label
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,NaN,NaN,NaN,NaN,6,5.0,0.783517,6,0,0,100.0,0.0,0.0,positive
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,NaN,NaN,NaN,NaN,5,3.4,0.542140,4,1,0,80.0,20.0,0.0,positive
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,NaN,NaN,NaN,NaN,5,4.4,0.545080,5,0,0,100.0,0.0,0.0,positive


Save outputs (CSV + Excel) and update runs.json

In [5]:
# === Save ===
# 1) Row-level scored reviews (CSV for downstream use if needed)
sent_csv = FINAL / "sentiment.csv"
reviews_scored[[
    "review_id","place_id","name","lat","lng","category",
    "rating","publish_time_utc","text","compound","sentiment_label"
]].to_csv(sent_csv, index=False)

# 2) Place-level Excel (similar to your shops_sentiment.xlsx, easy to demo)
xlsx_path = SENT_DIR / "shops_sentiment.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    grp.to_excel(writer, sheet_name="places", index=False)
    reviews_scored.head(2000).to_excel(writer, sheet_name="sample_reviews", index=False)

print(f"Saved:\n - {sent_csv}\n - {xlsx_path}")

# 3) Touch runs.json with a note
runs_path = FINAL/"runs.json"
try:
    runs = json.load(open(runs_path, "r", encoding="utf-8"))
except:
    runs = {}
runs["sentiment_last_built"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
with open(runs_path, "w", encoding="utf-8") as f:
    json.dump(runs, f, indent=2)
print("Updated runs.json")


Saved:
 - data/final/sentiment.csv
 - data/sentiment/shops_sentiment.xlsx
Updated runs.json
